# Exploratory Data Analysis

## Import Data and Libraries

In [ ]:
import pandas as pd
import numpy as np

# Load the dataset
df = pd.read_csv('./Flat prices.csv')

df

## Data Quality Assessment

In [ ]:
# Check for missing values
print("Missing Values:")
print(df.isnull().sum())

In [ ]:
# Check data types
print("\nData Types:")
print(df.dtypes)

In [ ]:
# Check for duplicates
print(f"\nDuplicate rows: {df.duplicated().sum()}")

In [ ]:
# Basic statistical summary
print("\nStatistical Summary:")
print(df.describe())

In [ ]:
# Check unique values for categorical columns
categorical_cols = ['town', 'flat_type', 'flat_model', 'storey_range', 'street_name', 'block']
for col in categorical_cols:
    if col in df.columns:
        print(f"\nUnique values in {col} with the lowest counts:")
        print(df[col].value_counts().tail(10))

# Data Cleaning

In [ ]:
# Create a copy for cleaning
df_clean = df.copy()

df_clean

## Duplicate Records

In [ ]:
# Remove duplicates
df_clean = df_clean.drop_duplicates()

print(f"After removing duplicates: {df_clean.shape}")
print(f"Duplicates removed: {df.shape[0] - df_clean.shape[0]}")

In [ ]:
df_clean

## Drop Redundant Rows

In [ ]:
# Drop 'month' and 'lease_commence_date' columns from df_clean
df_clean = df_clean.drop(columns=['month', 'lease_commence_date'], errors='ignore')
df_clean.head()

## Incorrect Data Types

### Remaining Lease Column

In [ ]:
# Convert remaining_lease to numeric
def extract_lease_years(lease_str):
    if pd.isna(lease_str):
        return np.nan
    if isinstance(lease_str, (int, float)):
        return float(lease_str)
    try:
        lease_str_lower = str(lease_str).lower()
        # Handle formats like "62 y 05 months" - extract the first number
        if any(keyword in lease_str_lower for keyword in ['year', 'y', 'years', 'yrs']):
            # Extract the first number (years) - works for "62 y 05 months"
            years = int(lease_str_lower.split()[0])
            return years
        else:
            return float(lease_str)
    except:
        return np.nan

df_clean['remaining_lease'] = df_clean['remaining_lease'].apply(extract_lease_years)
print(f"Remaining lease range: {df_clean['remaining_lease'].min():.0f} to {df_clean['remaining_lease'].max():.0f} years")
print(f"Remaining lease missing values after conversion: {df_clean['remaining_lease'].isnull().sum()}")


In [ ]:
df_clean

## Standardise Inconsistent Values

### Street Name Column

In [ ]:
# Sort street names alphabetically and print counts
street_value_counts = df_clean['street_name'].value_counts()
street_value_counts_sorted = street_value_counts.sort_index()

print("All street names (Alphabetical):")
for street, count in street_value_counts_sorted.items():
  print(f"  {street}: {count} records")

print(f"Total number of distinct values {len(street_value_counts_sorted)}")

In [ ]:
# Define mapping for street_name standardization
street_name_mapping = {
    # ANG MO KIO abbreviations
    'AMK AVE 10': 'ANG MO KIO AVE 10',
    'AMK AVE 1': 'ANG MO KIO AVE 1',
    'AMK AVE 2': 'ANG MO KIO AVE 2',
    'AMK AVE 3': 'ANG MO KIO AVE 3',
    'AMK AVE 4': 'ANG MO KIO AVE 4',
    'AMK AVE 5': 'ANG MO KIO AVE 5',
    'AMK AVE 6': 'ANG MO KIO AVE 6',
    'AMK AVE 8': 'ANG MO KIO AVE 8',
    'AMK AVE 9': 'ANG MO KIO AVE 9',
    
    # TOA PAYOH abbreviations
    'LOR 2 TP': 'LOR 2 TOA PAYOH',
    'LOR 7 TP': 'LOR 7 TOA PAYOH', 
    'LOR 8 TP': 'LOR 8 TOA PAYOH',
    
    # Directional abbreviations
    'BEDOK NTH AVE 1': 'BEDOK NORTH AVE 1',
    'BEDOK NTH AVE 2': 'BEDOK NORTH AVE 2',
    'BEDOK NTH AVE 3': 'BEDOK NORTH AVE 3',
    'BEDOK NTH AVE 4': 'BEDOK NORTH AVE 4',
    'BEDOK NTH RD': 'BEDOK NORTH RD',
    'BEDOK NTH ST 1': 'BEDOK NORTH ST 1',
    'BEDOK NTH ST 2': 'BEDOK NORTH ST 2',
    'BEDOK NTH ST 3': 'BEDOK NORTH ST 3',
    'BEDOK NTH ST 4': 'BEDOK NORTH ST 4',
    'BEDOK STH AVE 1': 'BEDOK SOUTH AVE 1',
    'BEDOK STH AVE 2': 'BEDOK SOUTH AVE 2',
    'BEDOK STH AVE 3': 'BEDOK SOUTH AVE 3',
    'BEDOK STH RD': 'BEDOK SOUTH RD',
    
    # CENTRAL abbreviations
    'BEDOK CTRL': 'BEDOK CENTRAL',
    'BT BATOK CTRL': 'BUKIT BATOK CENTRAL',
    'CHOA CHU KANG CTRL': 'CHOA CHU KANG CENTRAL',
    'HOUGANG CTRL': 'HOUGANG CENTRAL',
    'JURONG WEST CTRL 1': 'JURONG WEST CENTRAL 1',
    'JURONG WEST CTRL 3': 'JURONG WEST CENTRAL 3',
    'SENGKANG CTRL': 'SENGKANG CENTRAL',
    'SERANGOON CTRL': 'SERANGOON CENTRAL',
    'SERANGOON CTRL DR': 'SERANGOON CENTRAL DR',
    'TAMPINES CTRL 1': 'TAMPINES CENTRAL 1',
    'TAMPINES CTRL 7': 'TAMPINES CENTRAL 7',
    'TAMPINES CTRL 8': 'TAMPINES CENTRAL 8',
    'TOA PAYOH CTRL': 'TOA PAYOH CENTRAL',
    'YISHUN CTRL': 'YISHUN CENTRAL',
    'YISHUN CTRL 1': 'YISHUN CENTRAL 1',
    
    # Common abbreviations
    'C\'WEALTH AVE': 'COMMONWEALTH AVE',
    'C\'WEALTH AVE WEST': 'COMMONWEALTH AVE WEST',
    'C\'WEALTH CL': 'COMMONWEALTH CL',
    'C\'WEALTH CRES': 'COMMONWEALTH CRES',
    'C\'WEALTH DR': 'COMMONWEALTH DR',
    
    # BT -> BUKIT
    'BT BATOK EAST AVE 3': 'BUKIT BATOK EAST AVE 3',
    'BT BATOK EAST AVE 4': 'BUKIT BATOK EAST AVE 4',
    'BT BATOK EAST AVE 5': 'BUKIT BATOK EAST AVE 5',
    'BT BATOK EAST AVE 6': 'BUKIT BATOK EAST AVE 6',
    'BT BATOK WEST AVE 2': 'BUKIT BATOK WEST AVE 2',
    'BT BATOK WEST AVE 4': 'BUKIT BATOK WEST AVE 4',
    'BT BATOK WEST AVE 5': 'BUKIT BATOK WEST AVE 5',
    'BT BATOK WEST AVE 6': 'BUKIT BATOK WEST AVE 6',
    'BT BATOK WEST AVE 7': 'BUKIT BATOK WEST AVE 7',
    'BT BATOK WEST AVE 8': 'BUKIT BATOK WEST AVE 8',
    'BT BATOK WEST AVE 9': 'BUKIT BATOK WEST AVE 9',
    'BT MERAH CTRL': 'BUKIT MERAH CENTRAL',
    'BT MERAH LANE 1': 'BUKIT MERAH LANE 1',
    'BT MERAH VIEW': 'BUKIT MERAH VIEW',
    'BT PANJANG RING RD': 'BUKIT PANJANG RING RD',
    'BT PURMEI RD': 'BUKIT PURMEI RD',
    
    # JLN -> JALAN
    'JLN BAHAGIA': 'JALAN BAHAGIA',
    'JLN BATU': 'JALAN BATU',
    'JLN BERSEH': 'JALAN BERSEH',
    'JLN BT HO SWEE': 'JALAN BUKIT HO SWEE',
    'JLN BT MERAH': 'JALAN BUKIT MERAH',
    'JLN BT MERAH BT MERAH': 'JALAN BUKIT MERAH',  # Remove duplicate
    'JLN DAMAI': 'JALAN DAMAI',
    'JLN DUA': 'JALAN DUA',
    'JLN DUSUN': 'JALAN DUSUN',
    'JLN KAYU': 'JALAN KAYU',
    'JLN KLINIK': 'JALAN KLINIK',
    'JLN KUKOH': 'JALAN KUKOH',
    'JLN MA\'MOR': 'JALAN MA\'MOR',
    'JLN MEMBINA': 'JALAN MEMBINA',
    'JLN RAJAH': 'JALAN RAJAH',
    'JLN RUMAH TINGGI': 'JALAN RUMAH TINGGI',
    'JLN TECK WHYE': 'JALAN TECK WHYE',
    'JLN TENAGA': 'JALAN TENAGA',
    'JLN TENTERAM': 'JALAN TENTERAM',
    'JLN TIGA': 'JALAN TIGA',
    
    # Other common abbreviations
    'NTH BRIDGE RD': 'NORTH BRIDGE RD',
    'NEW MKT RD': 'NEW MARKET RD',
    'UPP CHANGI RD': 'UPPER CHANGI RD',
    'TG PAGAR PLAZA': 'TANJONG PAGAR PLAZA',
    
    # Remove duplicates and typos
    'CANTONMENT CL CL': 'CANTONMENT CL',
    'FERNVALE LINK LINK': 'FERNVALE LINK',
    'SENGKANG EAST WAY LINK': 'SENGKANG EAST WAY',
    'TAMPINES CTeRL 1': 'TAMPINES CENTRAL 1',
    'TAMPINES CTeRL 7': 'TAMPINES CENTRAL 7', 
    'TAMPINES CeRL 7': 'TAMPINES CENTRAL 7',
    'KIM TIAN RD Spore': 'KIM TIAN RD'
}

# Apply the mapping to standardize street_name
df_clean['street_name'] = df_clean['street_name'].replace(street_name_mapping)

print("After standardization:")
print(df_clean['street_name'].value_counts())
print(f"Total number of distinct values {len(df_clean['street_name'].value_counts())}")

# Sum counts of the removed distinct values (keys present in street_value_counts)
mapping_keys = set(street_name_mapping.keys())
present_keys = street_value_counts.index.intersection(mapping_keys)

removed_counts = street_value_counts.loc[present_keys]
affected_rows = int(removed_counts.sum())

# Optional: dataframe of affected rows (rows whose original street_name was one of the mapped keys)
affected_rows_df = df_clean[df_clean['street_name'].isin(present_keys)].copy()

print(f"Distinct mapped keys present: {len(present_keys)}")
print(removed_counts.sort_values(ascending=False))
print(f"Total rows affected by mapping: {affected_rows}")


In [ ]:
df_clean

## Check the Final Cleaned Dataset

In [ ]:
# Check for duplicate rows in df_clean
duplicate_count = df_clean.duplicated().sum()
print(f"Number of duplicate rows in df_clean: {duplicate_count}")

In [ ]:
df_clean = df_clean.drop_duplicates()
print(f"Rows after dropping duplicates: {len(df_clean):,}")

In [ ]:
# Check for missing values
print("Missing Values:")
print(df_clean.isnull().sum())

# Check data types
print("\nData Types:")
print(df_clean.dtypes)

# Basic statistical summary
print("\nStatistical Summary:")
print(df_clean.describe())

# Check unique values for categorical columns
categorical_cols = ['town', 'flat_type', 'flat_model', 'storey_range']
for col in categorical_cols:
    if col in df_clean.columns:
        print(f"\nUnique values in {col}:")
        print(df_clean[col].value_counts().head(10))

In [ ]:
df_clean

In [ ]:
# Print the number of unique values for each column in df_clean
for col in df_clean.columns:
  unique_count = df_clean[col].nunique(dropna=True)
  print(f"{col}: {unique_count} unique values")

# Geocoding

## Create Address Column

In [ ]:
df_clean['address'] = df_clean["block"].astype(str) + " " + df_clean["street_name"]
df_clean

## Use OneMap API to Get Latitude and Longitude

In [ ]:
import requests
import json

def get_onemap_token(email, password):
    """
    Get access token from OneMap API
    """
    auth_url = "https://www.onemap.gov.sg/api/auth/post/getToken"
    
    payload = {
        'email': email,
        'password': password
    }
    
    headers = {
        'Content-Type': 'application/json',
        'User-Agent': 'Your App Name'
    }
    
    try:
        response = requests.post(auth_url, json=payload, headers=headers)
        if response.status_code == 200:
            token_data = response.json()
            return token_data.get('access_token')
        else:
            print(f"Error getting token: {response.status_code}")
            return None
    except Exception as e:
        print(f"Exception getting token: {e}")
        return None

# Replace with your credentials
EMAIL = "andrewphw.24@ichat.sp.edu.sg"
PASSWORD = "mynameJeff01"

access_token = get_onemap_token(EMAIL, PASSWORD)
if access_token:
    print(f"Token obtained: {access_token}")
else:
    print("Failed to get token")

In [ ]:
# Function to get latitude and longitude for a given address
def get_lat_lon(address, token):
    url = (
        "https://www.onemap.gov.sg/api/common/elastic/search"
        f"?searchVal={address}&returnGeom=Y&getAddrDetails=Y&pageNum=1"
    )
    headers = {"Authorization": token}

    r = requests.get(url, headers=headers)
    if r.status_code != 200:
        return None, None

    data = r.json()
    if data["found"] == 0:
        return None, None

    result = data["results"][0]
    return result["LATITUDE"], result["LONGITUDE"]

In [ ]:
# -----------------------------
# Geocode UNIQUE addresses with checkpoints
# -----------------------------
# Extract unique addresses
unique_addresses = df_clean["address"].unique()
print("Total unique addresses:", len(unique_addresses))

token = access_token
address_cache = {}

for idx, addr in enumerate(unique_addresses, 1):  # start counting from 1
    lat, lon = get_lat_lon(addr, token)
    address_cache[addr] = (lat, lon)
    
    # Print checkpoint every 500 addresses
    if idx % 500 == 0 or idx == len(unique_addresses):
        print(f"Completed {idx}/{len(unique_addresses)} unique addresses")

# -----------------------------
# Map results back to full dataset
# -----------------------------
df_clean["latitude"] = df_clean["address"].map(lambda x: address_cache[x][0])
df_clean["longitude"] = df_clean["address"].map(lambda x: address_cache[x][1])

In [ ]:
df_clean.csv('"Flat prices geocoded.csv"')

# Basic Model Development

## Data Preparation

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib


In [4]:
df = pd.read_csv("data/Flat prices geocoded.csv")
print(df.head())


         town flat_type block        street_name storey_range  floor_area_sqm  \
0  ANG MO KIO    2 ROOM   406  ANG MO KIO AVE 10     10 TO 12            44.0   
1  ANG MO KIO    3 ROOM   108   ANG MO KIO AVE 4     01 TO 03            67.0   
2  ANG MO KIO    3 ROOM   602   ANG MO KIO AVE 5     01 TO 03            67.0   
3  ANG MO KIO    3 ROOM   465  ANG MO KIO AVE 10     04 TO 06            68.0   
4  ANG MO KIO    3 ROOM   601   ANG MO KIO AVE 5     01 TO 03            67.0   

       flat_model  remaining_lease  resale_price                address  \
0        Improved               61      232000.0  406 ANG MO KIO AVE 10   
1  New Generation               60      250000.0   108 ANG MO KIO AVE 4   
2  New Generation               62      262000.0   602 ANG MO KIO AVE 5   
3  New Generation               62      265000.0  465 ANG MO KIO AVE 10   
4  New Generation               62      265000.0   601 ANG MO KIO AVE 5   

   latitude   longitude  
0  1.362005  103.853880  
1  1.37096

In [5]:
df["location"] = df["town"]
basic_features = ["location", "flat_type", "floor_area_sqm", "remaining_lease"]
target = "resale_price"

X = df[basic_features]
y = df[target]


In [ ]:
print(df.head())

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

X_train.shape, X_test.shape


## Model Training

In [ ]:
categorical_cols = ["location", "flat_type"]
numeric_cols = ["floor_area_sqm", "remaining_lease"]

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
        ("num", "passthrough", numeric_cols),
    ]
)

rf_basic = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=250,
        max_depth=20,
        min_samples_leaf=2,
        n_jobs=-1,
        random_state=42
    ))
])


In [ ]:
rf_basic.fit(X_train, y_train)
print("Training complete.")


In [ ]:
# Extract the OneHotEncoder inside the pipeline
preprocessor = rf_basic.named_steps["preprocessor"]
if hasattr(preprocessor, "named_transformers_"):
	ohe = preprocessor.named_transformers_["cat"]
	# Get encoded column names
	encoded_cols = ohe.get_feature_names_out(["location", "flat_type"])
	print(encoded_cols)
	print(len(encoded_cols))
else:
	print("The preprocessor has not been fitted yet. Please fit the pipeline first.")


In [ ]:
preds = rf_basic.predict(X_test)

mae = mean_absolute_error(y_test, preds)
rmse = mean_squared_error(y_test, preds, squared=False)
r2 = r2_score(y_test, preds)

print("BASIC MODEL EVALUATION")
print("------------------------")
print("MAE :", mae)
print("RMSE:", rmse)
print("R²  :", r2)


In [ ]:
joblib.dump(rf_basic, "rf_model_basic.pkl")
print("Saved as rf_model_basic.pkl")


In [ ]:
import numpy as np

sample = pd.DataFrame([
    {"location": town, "flat_type": "4 ROOM", "floor_area_sqm": 90, "remaining_lease": 90}
    for town in df["location"].unique()
])

rf_basic.predict(sample)


# Precise Model Development

## Data Preparation

In [6]:
df = pd.read_csv("data/Flat prices geocoded.csv")
print(df.head())

         town flat_type block        street_name storey_range  floor_area_sqm  \
0  ANG MO KIO    2 ROOM   406  ANG MO KIO AVE 10     10 TO 12            44.0   
1  ANG MO KIO    3 ROOM   108   ANG MO KIO AVE 4     01 TO 03            67.0   
2  ANG MO KIO    3 ROOM   602   ANG MO KIO AVE 5     01 TO 03            67.0   
3  ANG MO KIO    3 ROOM   465  ANG MO KIO AVE 10     04 TO 06            68.0   
4  ANG MO KIO    3 ROOM   601   ANG MO KIO AVE 5     01 TO 03            67.0   

       flat_model  remaining_lease  resale_price                address  \
0        Improved               61      232000.0  406 ANG MO KIO AVE 10   
1  New Generation               60      250000.0   108 ANG MO KIO AVE 4   
2  New Generation               62      262000.0   602 ANG MO KIO AVE 5   
3  New Generation               62      265000.0  465 ANG MO KIO AVE 10   
4  New Generation               62      265000.0   601 ANG MO KIO AVE 5   

   latitude   longitude  
0  1.362005  103.853880  
1  1.37096

In [7]:
df["location"] = df["street_name"]
precise_features = [
    "location",
    "storey_range",
    "flat_type",
    "floor_area_sqm",
    "remaining_lease",
    "latitude",
    "longitude"
]

X = df[precise_features]
y = df[target]


In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

X_train.shape, X_test.shape


((72924, 7), (18232, 7))

## Model Training

In [17]:
categorical_cols = [
    "location",
    "storey_range",
    "flat_type"
]

numeric_cols = [
    "floor_area_sqm",
    "remaining_lease",
    "latitude",
    "longitude"
]

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
        ("num", "passthrough", numeric_cols)
    ]
)

rf_precise = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=50,
        max_depth=20,
        min_samples_leaf=2,
        n_jobs=-1,
        random_state=42
    ))
])


In [18]:
rf_precise.fit(X_train, y_train)
print("Training complete.")


Training complete.


In [19]:
preds = rf_precise.predict(X_test)

mae = mean_absolute_error(y_test, preds)
rmse = mean_squared_error(y_test, preds, squared=False)
r2 = r2_score(y_test, preds)

print("PRECISE MODEL EVALUATION")
print("------------------------")
print("MAE :", mae)
print("RMSE:", rmse)
print("R²  :", r2)


PRECISE MODEL EVALUATION
------------------------
MAE : 19965.2255501855
RMSE: 28199.225063475376
R²  : 0.966902988616855


In [20]:
joblib.dump(rf_precise, "rf_model_precise.pkl")
print("Saved as rf_model_precise.pkl")


Saved as rf_model_precise.pkl
